# Archivus – Chunked (Resumable) Upload

This notebook exercises the **resumable chunked upload** surface, meant for large files and flaky networks.

| Step | Endpoint | Method |
|------|----------|--------|
| 1 | `/storage/file/upload/chunk/init` | POST |
| 2 | `/storage/file/upload/chunk/part` | POST (multipart) |
| 3 | `/storage/file/upload/chunk/status` | GET |
| 4 | `/storage/file/upload/chunk/complete` | POST |
| 5 | `/storage/file/upload/chunk/abort` | POST |

**Flow:** `init` returns an `uploadId`. The client splits the file into fixed-size chunks and sends each with `part`. Chunk writes are **idempotent**, so a dropped connection just means re-sending the missing chunks — `status` tells you which indexes already landed. `complete` assembles them and persists the file through the normal storage path.

**Pre-requisites:**
- Server running on `http://localhost:8080`
- An admin user (the auth cell creates one if missing)

In [ ]:
import requests, json, io, hashlib

BASE = 'http://localhost:8080'

def show(resp):
    try:
        body = resp.json()
    except Exception:
        body = resp.text
    print(f'Status : {resp.status_code}')
    print(f'Body   : {json.dumps(body, indent=2)}')
    return body

token     = None
drive_id  = None
upload_id = None

## 1 · Auth setup

Register + login as an admin user to get a token and the owner drive ID.
A duplicate-register 400 is harmless if the user already exists.

In [ ]:
requests.post(f'{BASE}/auth/register', json={
    'username': 'samar', 'password': 'password12', 'pin': '123456',
    'email': 'samar@example.com', 'user_type': 'business', 'is_admin': True,
})

resp = requests.post(f'{BASE}/auth/login', json={'username': 'samar', 'pin': '123456'})
body = show(resp)
token = body['token']

resp = requests.get(f'{BASE}/auth/user/info', headers={'Authorization': f'Bearer {token}'})
body = resp.json()
drives = body.get('drives', [])
drive_id = drives[0]['DriveID'] if drives else None
print(f'\ndrive_id : {drive_id}')
assert drive_id, 'drive_id is None — register/login failed'

## 2 · Build a test file and split it into chunks

We synthesize a multi-megabyte payload in memory, keep its SHA-256 to verify the
round-trip later, and slice it into fixed-size chunks. Any chunk size works — the
server only cares that indexes cover `[0, totalChunks)`.

In [ ]:
CHUNK_SIZE = 1 << 20  # 1 MB per chunk

# ~5.5 MB of deterministic bytes so the SHA check is meaningful.
payload = (b'archivus-chunk-test-' * 300_000)
source_sha = hashlib.sha256(payload).hexdigest()

chunks = [payload[i:i + CHUNK_SIZE] for i in range(0, len(payload), CHUNK_SIZE)]
total_chunks = len(chunks)
FILENAME = 'bigfile.bin'

print(f'payload size : {len(payload):,} bytes')
print(f'chunk size   : {CHUNK_SIZE:,} bytes')
print(f'total chunks : {total_chunks}')
print(f'source sha256: {source_sha}')

## 3 · Init the upload session

Tell the server the filename, target drive/folder, total size and chunk count.
It returns a server-generated `uploadId` (a UUID) and an empty `received` list.

In [ ]:
resp = requests.post(f'{BASE}/storage/file/upload/chunk/init',
    headers={'Authorization': f'Bearer {token}'},
    json={
        'filename':    FILENAME,
        'driveId':     drive_id,
        'folderPath':  'uploads/large',
        'contentType': 'application/octet-stream',
        'size':        len(payload),
        'totalChunks': total_chunks,
    })
body = show(resp)
assert resp.status_code == 200, 'init failed'
upload_id = body['uploadId']
print(f'\nupload_id : {upload_id}')

## 4 · Upload chunks — but drop one, to simulate a network crash

We upload every chunk **except one** (index 2), pretending the connection died
before it landed. Each `part` request is `multipart/form-data` with `uploadId`,
`chunkIndex`, and the raw bytes in the `chunk` file field.

In [ ]:
DROPPED = 2  # pretend this chunk never made it

def send_chunk(index):
    return requests.post(f'{BASE}/storage/file/upload/chunk/part',
        headers={'Authorization': f'Bearer {token}'},
        data={'uploadId': upload_id, 'chunkIndex': str(index)},
        files=[('chunk', (f'{index}.part', io.BytesIO(chunks[index]), 'application/octet-stream'))])

for i in range(total_chunks):
    if i == DROPPED:
        print(f'chunk {i:>2} : SKIPPED (simulated drop)')
        continue
    resp = send_chunk(i)
    assert resp.status_code == 200, show(resp)
    print(f'chunk {i:>2} : {resp.status_code}  received={resp.json()["received"]}')

## 5 · Check status to find the missing chunk (resume point)

After a disconnect the client asks the server which indexes it already has, and
diffs against `[0, totalChunks)` to know what to re-send.

In [ ]:
resp = requests.get(f'{BASE}/storage/file/upload/chunk/status',
    headers={'Authorization': f'Bearer {token}'},
    params={'uploadId': upload_id})
body = show(resp)
assert resp.status_code == 200

received = set(body['received'])
missing = [i for i in range(total_chunks) if i not in received]
print(f'\nmissing chunks : {missing}')
assert body['complete'] is False, 'should NOT be complete yet'
assert missing == [DROPPED], f'expected only chunk {DROPPED} missing'
print('✓ server correctly reports the dropped chunk as missing')

## 6 · Re-send only the missing chunks

This is the whole point of resumability: we upload just the missing index, not
the entire file again. (Re-sending an already-received chunk is also safe —
writes are idempotent.)

In [ ]:
for i in missing:
    resp = send_chunk(i)
    assert resp.status_code == 200, show(resp)
    print(f'resent chunk {i} : received={resp.json()["received"]}')

resp = requests.get(f'{BASE}/storage/file/upload/chunk/status',
    headers={'Authorization': f'Bearer {token}'},
    params={'uploadId': upload_id})
body = resp.json()
assert body['complete'] is True, 'all chunks should now be present'
print('✓ all chunks received — ready to complete')

## 7 · Complete the upload

The server assembles the chunks in order and persists the file through the same
path as a direct upload (enforcing write access, overwrite/versioning, and DB
metadata). The staging directory is then cleaned up.

In [ ]:
resp = requests.post(f'{BASE}/storage/file/upload/chunk/complete',
    headers={'Authorization': f'Bearer {token}'},
    json={'uploadId': upload_id})
show(resp)
assert resp.status_code == 200, 'complete failed'
print('✓ upload completed')

## 8 · Verify the assembled file round-trips correctly

List the target folder to find the file's ID, download it, and confirm the
SHA-256 matches the original payload — proving the chunks were reassembled in the
right order with no corruption.

In [ ]:
resp = requests.post(f'{BASE}/storage/files',
    headers={'Authorization': f'Bearer {token}'},
    json={'path': 'uploads/large', 'driveId': drive_id})
body = show(resp)

file_id = None
for e in body.get('files', []):
    if not e.get('IsDir') and e.get('Name') == FILENAME:
        file_id = e['ID']
        break
print(f'\nfile_id : {file_id}')
assert file_id, f'{FILENAME} not found in uploads/large'

In [ ]:
resp = requests.get(f'{BASE}/storage/file/download',
    headers={'Authorization': f'Bearer {token}'},
    params={'fileId': file_id, 'driveId': drive_id})
assert resp.status_code == 200, show(resp)

downloaded_sha = hashlib.sha256(resp.content).hexdigest()
print(f'downloaded size : {len(resp.content):,} bytes (expected {len(payload):,})')
print(f'downloaded sha  : {downloaded_sha}')
print(f'source sha      : {source_sha}')
assert len(resp.content) == len(payload), 'size mismatch'
assert downloaded_sha == source_sha, 'content mismatch — chunks reassembled incorrectly'
print('✓ round-trip verified: assembled file is byte-identical to the source')

## 9 · Abort discards an in-progress session

Start a fresh session, upload a chunk, then abort it. A follow-up `status` call
should 404 because the staging directory is gone.

In [ ]:
resp = requests.post(f'{BASE}/storage/file/upload/chunk/init',
    headers={'Authorization': f'Bearer {token}'},
    json={'filename': 'throwaway.bin', 'driveId': drive_id, 'folderPath': 'uploads/large',
          'size': len(payload), 'totalChunks': total_chunks})
abort_id = resp.json()['uploadId']

send = requests.post(f'{BASE}/storage/file/upload/chunk/part',
    headers={'Authorization': f'Bearer {token}'},
    data={'uploadId': abort_id, 'chunkIndex': '0'},
    files=[('chunk', ('0.part', io.BytesIO(chunks[0]), 'application/octet-stream'))])
print(f'seeded one chunk : {send.status_code}')

resp = requests.post(f'{BASE}/storage/file/upload/chunk/abort',
    headers={'Authorization': f'Bearer {token}'},
    json={'uploadId': abort_id})
show(resp)
assert resp.status_code == 200

resp = requests.get(f'{BASE}/storage/file/upload/chunk/status',
    headers={'Authorization': f'Bearer {token}'},
    params={'uploadId': abort_id})
print(f'status after abort : {resp.status_code}  (expect 404)')
assert resp.status_code == 404, 'aborted session should no longer exist'
print('✓ aborted session was discarded')

## 10 · Sessions are private to their owner

A different user must not be able to see or touch someone else's upload session.
We create a read-access user and confirm their `status` call 404s (the server
hides existence from non-owners).

In [ ]:
# Fresh session owned by 'samar'
resp = requests.post(f'{BASE}/storage/file/upload/chunk/init',
    headers={'Authorization': f'Bearer {token}'},
    json={'filename': 'private.bin', 'driveId': drive_id, 'folderPath': 'uploads/large',
          'size': len(payload), 'totalChunks': total_chunks})
private_id = resp.json()['uploadId']

# Create a read-access user in the same drive
invite = requests.post(f'{BASE}/auth/drive/invite',
    headers={'Authorization': f'Bearer {token}'},
    json={'drive_id': drive_id, 'access': 'read'}).json()['invite_code']
requests.post(f'{BASE}/auth/register', json={
    'username': 'readonly', 'password': 'readonly12', 'pin': '111111',
    'email': 'ro@example.com', 'user_type': 'business', 'is_admin': False,
    'invite_code': invite,
})
ro_token = requests.post(f'{BASE}/auth/login', json={'username': 'readonly', 'pin': '111111'}).json()['token']

resp = requests.get(f'{BASE}/storage/file/upload/chunk/status',
    headers={'Authorization': f'Bearer {ro_token}'},
    params={'uploadId': private_id})
print(f'other user status : {resp.status_code}  (expect 404)')
assert resp.status_code == 404, 'a non-owner must not access the session'
print('✓ session is not visible to other users')

# cleanup
requests.post(f'{BASE}/storage/file/upload/chunk/abort',
    headers={'Authorization': f'Bearer {token}'}, json={'uploadId': private_id})